## Second Innings

In [4]:
import pandas as pd
import joblib

In [5]:
delivery = pd.read_csv('..\data\deliveries_new.csv')
matches = pd.read_csv('..\data\matches_new.csv')

In [6]:
match_df = matches[['id','venue','winner','target_runs']].copy()

match_df.rename(columns={
    'id':'match_id',
    'target_runs':'target'
}, inplace=True)

In [7]:
delivery = match_df.merge(delivery,on='match_id')

# Only second innings
delivery = delivery[delivery['inning'] == 2]

# FIX: overs start from 0 in new dataset
delivery['over'] = delivery['over'] + 1

delivery = delivery.sort_values(['match_id','over','ball'])

In [8]:
delivery.columns

Index(['match_id', 'venue', 'winner', 'target', 'inning', 'batting_team',
       'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker',
       'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket',
       'player_dismissed', 'dismissal_kind', 'fielder'],
      dtype='object')

In [9]:
match_df.shape

(581, 4)

In [10]:
match_df.columns

Index(['match_id', 'venue', 'winner', 'target'], dtype='object')

In [11]:
delivery.columns

Index(['match_id', 'venue', 'winner', 'target', 'inning', 'batting_team',
       'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker',
       'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket',
       'player_dismissed', 'dismissal_kind', 'fielder'],
      dtype='object')

In [12]:
delivery['current_score'] = (
    delivery.groupby('match_id')['total_runs']
    .cumsum()
)

In [13]:
delivery['legal_ball'] = delivery['extra_runs'] == 0

delivery['balls_bowled'] = (
    delivery[delivery['legal_ball']]
    .groupby('match_id')
    .cumcount() + 1
)

delivery['balls_bowled'] = (
    delivery.groupby('match_id')['balls_bowled']
    .ffill()
)

# Balls left
delivery['balls_left'] = 120 - delivery['balls_bowled']

In [14]:
delivery['wickets'] = delivery.groupby('match_id')['is_wicket'].cumsum()
delivery['wickets_left'] = 10 - delivery['wickets']

In [15]:
delivery['runs_left'] = delivery['target'] + 1 - delivery['current_score']


In [16]:
delivery.head()

,match_id,venue,winner,target,inning,batting_team,bowling_team,over,ball,batter,...,player_dismissed,dismissal_kind,fielder,current_score,legal_ball,balls_bowled,balls_left,wickets,wickets_left,runs_left
122,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,1,RV Uthappa,...,NaN,NaN,NaN,1,True,1.0,119.0,0,10,169.0
123,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,2,G Gambhir,...,NaN,NaN,NaN,2,False,1.0,119.0,0,10,168.0
124,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,3,G Gambhir,...,NaN,NaN,NaN,3,False,1.0,119.0,0,10,167.0
125,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,4,RV Uthappa,...,NaN,NaN,NaN,3,True,2.0,118.0,0,10,167.0
126,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,5,RV Uthappa,...,NaN,NaN,NaN,3,True,3.0,117.0,0,10,167.0


In [17]:
delivery = delivery[
    (delivery['runs_left'] > 0) &
    (delivery['balls_left'] > 0) &
    (delivery['wickets_left'] > 0)
]

In [18]:
delivery

,match_id,venue,winner,target,inning,batting_team,bowling_team,over,ball,batter,...,player_dismissed,dismissal_kind,fielder,current_score,legal_ball,balls_bowled,balls_left,wickets,wickets_left,runs_left
122,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,1,RV Uthappa,...,NaN,NaN,NaN,1,True,1.0,119.0,0,10,169.0
123,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,2,G Gambhir,...,NaN,NaN,NaN,2,False,1.0,119.0,0,10,168.0
124,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,3,G Gambhir,...,NaN,NaN,NaN,3,False,1.0,119.0,0,10,167.0
125,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,4,RV Uthappa,...,NaN,NaN,NaN,3,True,2.0,118.0,0,10,167.0
126,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,5,RV Uthappa,...,NaN,NaN,NaN,3,True,3.0,117.0,0,10,167.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138896,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,10,5,SS Iyer,...,NaN,NaN,NaN,110,True,56.0,64.0,2,8,5.0
138897,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,10,6,VR Iyer,...,NaN,NaN,NaN,111,True,57.0,63.0,2,8,4.0
138898,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,11,1,VR Iyer,...,NaN,NaN,NaN,112,True,58.0,62.0,2,8,3.0
138899,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,11,2,SS Iyer,...,NaN,NaN,NaN,113,True,59.0,61.0,2,8,2.0


In [19]:
delivery

,match_id,venue,winner,target,inning,batting_team,bowling_team,over,ball,batter,...,player_dismissed,dismissal_kind,fielder,current_score,legal_ball,balls_bowled,balls_left,wickets,wickets_left,runs_left
122,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,1,RV Uthappa,...,NaN,NaN,NaN,1,True,1.0,119.0,0,10,169.0
123,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,2,G Gambhir,...,NaN,NaN,NaN,2,False,1.0,119.0,0,10,168.0
124,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,3,G Gambhir,...,NaN,NaN,NaN,3,False,1.0,119.0,0,10,167.0
125,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,4,RV Uthappa,...,NaN,NaN,NaN,3,True,2.0,118.0,0,10,167.0
126,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,5,RV Uthappa,...,NaN,NaN,NaN,3,True,3.0,117.0,0,10,167.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138896,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,10,5,SS Iyer,...,NaN,NaN,NaN,110,True,56.0,64.0,2,8,5.0
138897,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,10,6,VR Iyer,...,NaN,NaN,NaN,111,True,57.0,63.0,2,8,4.0
138898,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,11,1,VR Iyer,...,NaN,NaN,NaN,112,True,58.0,62.0,2,8,3.0
138899,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,11,2,SS Iyer,...,NaN,NaN,NaN,113,True,59.0,61.0,2,8,2.0


In [20]:
delivery['crr'] = delivery['current_score'] * 6 / delivery['balls_bowled'].replace(0,1)

delivery['rrr'] = delivery['runs_left'] * 6 / delivery['balls_left'].replace(0,1)

In [21]:
delivery['result'] = (delivery['batting_team'] == delivery['winner']).astype(int)

In [22]:
delivery

,match_id,venue,winner,target,inning,batting_team,bowling_team,over,ball,batter,...,current_score,legal_ball,balls_bowled,balls_left,wickets,wickets_left,runs_left,crr,rrr,result
122,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,1,RV Uthappa,...,1,True,1.0,119.0,0,10,169.0,6.000000,8.521008,1
123,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,2,G Gambhir,...,2,False,1.0,119.0,0,10,168.0,12.000000,8.470588,1
124,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,3,G Gambhir,...,3,False,1.0,119.0,0,10,167.0,18.000000,8.420168,1
125,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,4,RV Uthappa,...,3,True,2.0,118.0,0,10,167.0,9.000000,8.491525,1
126,829705,Eden Gardens,KKR,169.0,2,KKR,MI,1,5,RV Uthappa,...,3,True,3.0,117.0,0,10,167.0,6.000000,8.564103,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138896,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,10,5,SS Iyer,...,110,True,56.0,64.0,2,8,5.0,11.785714,0.468750,1
138897,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,10,6,VR Iyer,...,111,True,57.0,63.0,2,8,4.0,11.684211,0.380952,1
138898,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,11,1,VR Iyer,...,112,True,58.0,62.0,2,8,3.0,11.586207,0.290323,1
138899,1426312,MA Chidambaram Stadium,KKR,114.0,2,KKR,SRH,11,2,SS Iyer,...,113,True,59.0,61.0,2,8,2.0,11.491525,0.196721,1


In [23]:
# print(delivery[['batting_team', 'winner']].drop_duplicates().head(20))

In [24]:
delivery['result'].value_counts()

result
0    33473
1    33114
Name: count, dtype: int64

In [25]:
final_df = delivery[[
    'batting_team',
    'bowling_team',
    'venue',
    'runs_left',
    'balls_left',
    'wickets_left',
    'target',
    'crr',
    'rrr',
    'result'
]]

In [26]:
#encoding
final_df = pd.get_dummies(final_df, columns=['batting_team', 'bowling_team', 'venue'])

In [27]:
from sklearn.model_selection import train_test_split

X = final_df.drop('result', axis=1)
y = final_df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
from sklearn.linear_model import LogisticRegression

second_innings = LogisticRegression(
    max_iter=2000,
    solver='lbfgs'
)
second_innings.fit(X_train, y_train)

joblib.dump(second_innings, '..\models\second_innings.pkl')
joblib.dump(X.columns.tolist(),"..\models\second_features.pkl")


c:\Users\hazra\anaconda3\envs\AI_ML_env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


['..\\models\\second_features.pkl']

In [29]:
from sklearn.metrics import accuracy_score

y_pred = second_innings.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7979426340291335


In [30]:
second_innings.predict_proba(X_test)

array([[0.57403003, 0.42596997],
       [0.73010937, 0.26989063],
       [0.15440247, 0.84559753],
       ...,
       [0.84023009, 0.15976991],
       [0.83358134, 0.16641866],
       [0.81699494, 0.18300506]], shape=(13318, 2))

In [31]:
powerplay = delivery[(delivery['over'] <= 6) & (delivery['ball'] == 6)]
middle = delivery[(delivery['over'] > 6) & (delivery['over'] <= 15) & (delivery['ball'] == 6)]
death = delivery[delivery['over'] > 15]